# XOR Logic Gate Network
This project focuses on designing a simple neural network to replicate the XOR gate truth table.

## Background

### Perceptron Problem
Although Perceptron networks have an activation function to non-linearise patternisation, the raw boundary itself is still a single line in n-dimensional space. For the common logic gates, that single line is enough to separate the true and false combinations of their truth tables, but the XOR gate is more complicated: it is impossible to separate its input combinations into two distinct regions using a single line. This issue is generalised for virtually all other cases, since models tend to be far more complex than simple logic gate truth tables. Therefore, a multi-neuron neural network is virtually always required.

### Hidden Layers
A non-linear neural network contains hidden layers: layers of neurons that fall in between the input and output layers of the network. These layers contain boundary lines only used by and visible to the network, where each neuron in each layer collects all the outputs from the previous layer as its input vector — continuing the same pathway logic for the subsequent hidden layers.

For instance, consider an arbitrary hidden layer neuron, `N`. Each neuron in the network takes in 2 inputs, so `N` will take the output from the previous neurons <code>n<sub>1</sub></code> and <code>n<sub>2</sub></code>, apply its traditional neuron mathematics, then feed its output to all the succeeding neurons in the next hidden layer.

### Backpropagation
In a neural network, one can only determine the error in the final result compared to the expected output at the last neuron — the output layer. From here, the assumption is made that all neurons in the network are partially blamed for the error resulting in the output. This is achieved through the chain rule in differentiation.

For example, consider a network with two inputs, <code>x<sub>1</sub></code> and <code>x<sub>2</sub></code>, that are fed into 1 hidden neuron, where <code>y<sub>hidden</sub> = activation_function(w<sub>1</sub>x<sub>1</sub> + w<sub>2</sub>x<sub>2</sub> + b<sub>hidden</sub>)</code>. <code>y<sub>hidden</sub></code> is then passed into the output neuron, where <code>y<sub>output</sub> = activation_function(w<sub>3</sub>y<sub>hidden</sub> + b<sub>output</sub>)</code>.

To compute the loss and make adjustments, <code>w<sub>i</sub> ← w<sub>i</sub> - η ∂Loss/∂w<sub>i</sub></code> is performed for all <code>w<sub>i</sub></code>; in this case, <code>∂Loss/∂w<sub>1</sub> = (∂Loss/∂y<sub>output</sub>)(∂y<sub>output</sub>/∂y<sub>hidden</sub>)(∂y<sub>hidden</sub>/∂w<sub>1</sub>)</code>, <code>∂Loss/∂w<sub>2</sub> = (∂Loss/∂y<sub>output</sub>)(∂y<sub>output</sub>/∂y<sub>hidden</sub>)(∂y<sub>hidden</sub>/∂w<sub>2</sub>)</code>, and <code>∂Loss/∂w<sub>3</sub> = (∂Loss/∂y<sub>output</sub>)(∂y<sub>output</sub>/∂w<sub>3</sub>)</code>.

## XOR Logic Gate

### XOR Logic Gate Breakdown
Identifying the exact number of neurons needed in the hidden layer is difficult, but for the 2-field XOR logic gate, having 2 neurons in the hidden field is sufficient. The XOR truth table can be separated into distinct regions correctly using 2 boundary lines. Therefore, the XOR logic gate network will utilise 2 hidden neurons and 1 output neuron. 

The network will utilise the Sigmoid function, <code>S(x) = 1/(1+e<sup>-x</sup>)</code> as the activation function for it is best for binary classification.

The network will utilise the simple Mean-Squared Error function, <code>MSE = 1/2(y<sub>expected</sub> - y<sub>output</sub>)<sup>2</sup></code>.


### XOR Logic Gate Design

In [66]:
# Python Dependencies
import numpy as np

In [67]:
# Neuron Class Definition
class Neuron:
    def __init__(self, num_vars):
        self.weights = np.random.randn(num_vars)
        self.bias = np.random.randn()

    # Activation Function: Sigmoid Function
    def activation_function(self, linear_prediction):
        return 1/(1+np.e**(-linear_prediction))
    
    def der_activation_function(self, input):
        return np.e**(-input) / (1 + np.e**(-input))**2
        
    def predict(self, inputs):
        return self.activation_function(np.dot(inputs, self.weights) + self.bias)

# Neural Network Class Definition
class Neural_Network:
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
        self.hidden1 = Neuron(2)
        self.hidden2 = Neuron(2)
        self.output = Neuron(2)

    def predict(self, input):
        hidden_output = np.array([self.hidden1.predict(input), self.hidden2.predict(input)])
        return round(self.output.predict(hidden_output))
    
    def train(self, input_vector, expected_output_vector, epochs=10000):
        for _ in range(epochs):
            for input, output in zip(input_vector, expected_output_vector):
                prediction = self.predict(input) # Loss Function: MSE
                hidden_output = np.array([self.hidden1.predict(input), self.hidden2.predict(input)])
                error = prediction - output

                # By Chain Rule
                output_error = error * self.output.der_activation_function(np.dot(hidden_output, self.output.weights) + self.output.bias)
                hidden_error1 = output_error * self.output.weights[0] * self.hidden1.der_activation_function(np.dot(input, self.hidden1.weights) + self.hidden1.bias)
                hidden_error2 = output_error * self.output.weights[1] * self.hidden2.der_activation_function(np.dot(input, self.hidden2.weights) + self.hidden2.bias)

                # Weight Adjustments
                for i in range(2):
                    self.output.weights[i] -= self.learning_rate*output_error*hidden_output[i]
                    self.hidden1.weights[i] -= self.learning_rate*hidden_error1*input[i]
                    self.hidden2.weights[i] -= self.learning_rate*hidden_error2*input[i]

                # Bias Adjustments
                self.output.bias -= self.learning_rate*output_error
                self.hidden1.bias -= self.learning_rate*hidden_error1
                self.hidden2.bias -= self.learning_rate*hidden_error2


### XOR Logic Gate Training

In [68]:
# Training
XOR_input = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

XOR_output = np.array([0, 1, 1, 0])

XOR_gate = Neural_Network()

XOR_gate.train(XOR_input, XOR_output, 100000)

### XOR Logic Gate Prediction

In [69]:
# Testing Output
for i in [[0, 0], [0, 1], [1, 0], [1, 1]]:
    print(i[0], "XOR", i[1], "=", XOR_gate.predict(i))

0 XOR 0 = 1
0 XOR 1 = 1
1 XOR 0 = 0
1 XOR 1 = 0
